# HSDS Pipeline Improvements — Testing & Implementation Guide

This notebook walks through every change made to the HSDS pipeline, explaining:
- **Where** to apply each change (locally, pgAdmin, AWS Console)
- **How** to test it produces identical output to current production
- **When** to deploy it to production

### Prerequisites
- Access to pgAdmin connected to the production RDS instance
- AWS Console access (Step Functions, Batch, ECR, Lambda, EventBridge, SNS)
- GitHub repo access with ability to add secrets
- Local `.env` file configured

---
## P1: Mastercard Performance

Three independent changes that each speed up the Mastercard pipeline. Test each separately.

### P1a: Database Indexes (pgAdmin)

**What:** Add indexes on `quad_id`, `yr`/`wk`, and `count_date` columns across staging, clean, raw, lookup, and 3-hourly tables.

**Where:** pgAdmin → run against production database

**Risk:** Zero — indexes are additive, they don't change data or query results. `CREATE INDEX IF NOT EXISTS` is safe to re-run.

**Steps:**
1. Open pgAdmin, connect to the production RDS
2. Open a new Query Tool
3. Copy and run the contents of `highstreets/sql/queries/mcard/create_mcard_indexes.sql`
4. Verify indexes were created (see verification cell below)

**Important:** Run this BEFORE testing any other Mastercard changes — the indexes will speed up the test runs too.

In [ ]:
# Read the CORRECTED index SQL file to see what will be applied
# Columns: quad_id (not quadkey), yr/wk (not week_start_date), count_date (not txn_date)
with open('../highstreets/sql/queries/mcard/create_mcard_indexes.sql') as f:
    print(f.read())

In [ ]:
# After running in pgAdmin, verify indexes exist
import os
import pandas as pd
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

load_dotenv()

engine = create_engine(
    f"postgresql+psycopg2://{os.getenv('PG_USER')}:{os.getenv('PG_PASSWORD')}@"
    f"{os.getenv('PG_HOST')}:{os.getenv('PG_PORT')}/{os.getenv('PG_DATABASE')}"
)

verify_sql = """
SELECT indexname, tablename
FROM pg_indexes
WHERE schemaname = 'gisapdata'
  AND (indexname LIKE 'idx_mcard%'
   OR indexname LIKE 'idx_mrli%'
   OR indexname LIKE 'idx_hs_%'
   OR indexname LIKE 'idx_bid_%'
   OR indexname LIKE 'idx_tc_%'
   OR indexname LIKE 'idx_bespoke_%'
   OR indexname LIKE 'idx_borough_%'
   OR indexname LIKE 'idx_caz_%'
   OR indexname LIKE 'idx_msoa_%'
   OR indexname LIKE 'idx_io_%')
ORDER BY tablename;
"""
with engine.connect() as conn:
    indexes = pd.read_sql(text(verify_sql), conn)
print(f"Found {len(indexes)} indexes:")
display(indexes)

In [ ]:
# Detect duplicate/redundant indexes — run AFTER creating all indexes
# Shows the actual indexed columns for every index, grouped by table,
# and flags when one index is a prefix-subset of another on the same table.

duplicate_check_sql = """
SELECT
    t.relname AS tablename,
    i.relname AS indexname,
    pg_get_indexdef(ix.indexrelid) AS indexdef,
    ix.indisunique AS is_unique,
    pg_relation_size(ix.indexrelid) AS index_size_bytes,
    pg_size_pretty(pg_relation_size(ix.indexrelid)) AS index_size
FROM pg_index ix
JOIN pg_class t ON t.oid = ix.indrelid
JOIN pg_class i ON i.oid = ix.indexrelid
JOIN pg_namespace n ON n.oid = t.relnamespace
WHERE n.nspname = 'gisapdata'
  AND i.relname LIKE 'idx_%'
ORDER BY t.relname, i.relname;
"""

with engine.connect() as conn:
    all_indexes = pd.read_sql(text(duplicate_check_sql), conn)

print(f"Total custom indexes: {len(all_indexes)}\n")

# Show full definitions grouped by table
for table, group in all_indexes.groupby("tablename"):
    if len(group) > 1:
        print(f"=== {table} ({len(group)} indexes) ===")
        for _, row in group.iterrows():
            uniq = " [UNIQUE]" if row["is_unique"] else ""
            print(f"  {row['indexname']}{uniq}  ({row['index_size']})")
            # Extract just the column part from the full CREATE INDEX statement
            defn = row["indexdef"]
            col_start = defn.find("(")
            if col_start > 0:
                print(f"    -> columns: {defn[col_start:]}")
        print()

# Detect prefix-subset duplicates
import re

def extract_columns(indexdef):
    """Extract column list from CREATE INDEX ... (col1, col2, ...)"""
    match = re.search(r'\((.+)\)$', indexdef)
    if match:
        return [c.strip() for c in match.group(1).split(",")]
    return []

print("=" * 60)
print("DUPLICATE / REDUNDANT INDEX ANALYSIS")
print("=" * 60)

found_any = False
for table, group in all_indexes.groupby("tablename"):
    if len(group) < 2:
        continue
    rows = group.to_dict("records")
    for i, a in enumerate(rows):
        for j, b in enumerate(rows):
            if i >= j:
                continue
            cols_a = extract_columns(a["indexdef"])
            cols_b = extract_columns(b["indexdef"])
            if not cols_a or not cols_b:
                continue

            # Exact duplicate
            if cols_a == cols_b:
                found_any = True
                print(f"\n  EXACT DUPLICATE on {table}:")
                print(f"    {a['indexname']}  ->  {', '.join(cols_a)}")
                print(f"    {b['indexname']}  ->  {', '.join(cols_b)}")
                if a["is_unique"] and not b["is_unique"]:
                    print(f"    RECOMMENDATION: DROP {b['indexname']} (non-unique copy of unique index)")
                elif b["is_unique"] and not a["is_unique"]:
                    print(f"    RECOMMENDATION: DROP {a['indexname']} (non-unique copy of unique index)")
                else:
                    print(f"    RECOMMENDATION: DROP {b['indexname']} (keep the older one)")

            # Prefix subset: a's columns are a leading prefix of b's
            elif cols_a == cols_b[:len(cols_a)] and len(cols_a) < len(cols_b):
                if not a["is_unique"]:
                    found_any = True
                    print(f"\n  REDUNDANT (prefix subset) on {table}:")
                    print(f"    {a['indexname']}  ->  {', '.join(cols_a)}")
                    print(f"    {b['indexname']}  ->  {', '.join(cols_b)}")
                    print(f"    RECOMMENDATION: DROP {a['indexname']} (subsumed by {b['indexname']})")

            elif cols_b == cols_a[:len(cols_b)] and len(cols_b) < len(cols_a):
                if not b["is_unique"]:
                    found_any = True
                    print(f"\n  REDUNDANT (prefix subset) on {table}:")
                    print(f"    {b['indexname']}  ->  {', '.join(cols_b)}")
                    print(f"    {a['indexname']}  ->  {', '.join(cols_a)}")
                    print(f"    RECOMMENDATION: DROP {b['indexname']} (subsumed by {a['indexname']})")

if not found_any:
    print("\n  No duplicates or redundant indexes found.")

In [6]:
# Check query plans for BOTH a bare join and a production-realistic filtered join
# Compare to see how much the WHERE clause + composite index helps

print("=" * 70)
print("TEST 1: Bare JOIN (no WHERE) — baseline, scans entire staging table")
print("=" * 70)
explain_bare = """
EXPLAIN ANALYZE
SELECT COUNT(*)
FROM gisapdata.econ_busyness_mcard_stg_18_zoom s
JOIN gisapdata.econ_busyness_mcard_highstreets_quad_lookup l
  ON s.quad_id::BIGINT = l.quad_id::BIGINT;
"""
with engine.connect() as conn:
    plan1 = pd.read_sql(text(explain_bare), conn)
for row in plan1.iloc[:, 0]:
    print(row)

print("\n" + "=" * 70)
print("TEST 2: Production-realistic query (mirrors highstreet_weekly_query.sql)")
print("=" * 70)
explain_prod = """
EXPLAIN ANALYZE
SELECT l.highstreet_id, l.highstreet_name, s.yr, s.wk,
       SUM(CASE WHEN s.weekday_weekend = 'weekdays' AND s.industry = 'Total Retail'
           THEN s.txn_amt ELSE 0 END) AS txn_amt_wd_retail
FROM gisapdata.econ_busyness_mcard_stg_18_zoom s
JOIN gisapdata.econ_busyness_mcard_highstreets_quad_lookup l
  ON s.quad_id::BIGINT = l.quad_id::BIGINT
WHERE s.industry IN ('Total Retail', 'Total Apparel', 'Eating Places')
  AND s.segment = 'Overall'
  AND s.geo_name = 'London'
GROUP BY l.highstreet_id, l.highstreet_name, s.yr, s.wk
ORDER BY s.yr, s.wk;
"""
with engine.connect() as conn:
    plan2 = pd.read_sql(text(explain_prod), conn)
for row in plan2.iloc[:, 0]:
    print(row)

print("\n" + "=" * 70)
print("COMPARE: Look for 'Index Scan' or 'Bitmap Index Scan' in Test 2")
print("vs 'Seq Scan' in Test 1. Test 2 should be significantly faster.")
print("=" * 70)

Finalize Aggregate  (cost=44137659.47..44137659.48 rows=1 width=8) (actual time=18682.033..18684.142 rows=1 loops=1)
  ->  Gather  (cost=44137659.25..44137659.46 rows=2 width=8) (actual time=17968.750..18684.132 rows=3 loops=1)
        Workers Planned: 2
        Workers Launched: 2
        ->  Partial Aggregate  (cost=44136659.25..44136659.26 rows=1 width=8) (actual time=17750.349..17750.352 rows=1 loops=3)
              ->  Merge Join  (cost=2855535.19..40402408.47 rows=1493700313 width=0) (actual time=13428.010..17393.393 rows=6218376 loops=3)
                    Merge Cond: (((s.quad_id)::bigint) = ((l.quad_id)::bigint))
                    ->  Sort  (cost=2853403.94..2887464.69 rows=13624302 width=18) (actual time=13412.183..15000.590 rows=10899308 loops=3)
                          Sort Key: ((s.quad_id)::bigint)
                          Sort Method: external merge  Disk: 414272kB
                          Worker 0:  Sort Method: external merge  Disk: 484472kB
                   

In [ ]:
# ============================================================
# BACKUP: Snapshot Mastercard output tables BEFORE testing changes
# Run this ONCE before running the modified mcard_weekly.py locally
# ============================================================
# This creates _backup copies of the TRUNCATE+LOAD output tables.
# If anything goes wrong, you can restore from these.

backup_tables = [
    "econ_busyness_mcard_bids_txn",
    "econ_busyness_mcard_highstreets_txn",
    "econ_busyness_mcard_towncentres_txn",
    "econ_busyness_mcard_boroughs_txn",
    "econ_busyness_mcard_caz_txn",
    "econ_busyness_mcard_msoas_txn",
    "econ_busyness_mcard_bespoke_txn",
    "econ_busyness_mcard_inner_outer_txn",
    "econ_busyness_mcard_bids_yoy",
    "econ_busyness_mcard_highstreets_yoy",
    "econ_busyness_mcard_towncentres_yoy",
    "econ_busyness_mcard_boroughs_yoy",
    "econ_busyness_mcard_caz_yoy",
    "econ_busyness_mcard_msoas_yoy",
    "econ_busyness_mcard_bespoke_yoy",
    "econ_busyness_mcard_inner_outer_yoy",
    "econ_busyness_mcard_txn",
    "econ_busyness_mcard_yoy",
]

with engine.connect() as conn:
    for table in backup_tables:
        backup = f"{table}_backup_20260720"
        conn.execute(text(f"DROP TABLE IF EXISTS gisapdata.{backup}"))
        conn.execute(text(
            f"CREATE TABLE gisapdata.{backup} AS SELECT * FROM gisapdata.{table}"
        ))
        count = conn.execute(text(
            f"SELECT COUNT(*) FROM gisapdata.{backup}"
        )).scalar()
        print(f"  {table} -> {backup} ({count:,} rows)")
    conn.commit()

print("\nBackup complete. To restore any table:")
print("  DROP TABLE gisapdata.{table};")
print("  ALTER TABLE gisapdata.{table}_backup_20260720 RENAME TO {table};")

### P1b: `method='multi'` on `append_chunk` (Local code change)

**What:** Changed `datawriter.py` so that `append_chunk()` and `append_data_without_check()` use `method='multi', chunksize=5000` instead of default row-by-row INSERT.

**Where:** Already applied in local code (`highstreets/data_source_sink/datawriter.py`)

**Risk:** Very low — same SQL output, just batched. The `truncate_and_load_to_postgres` method already uses this pattern successfully.

**How to test:** Compare raw file processing speed and verify identical row counts.

In [7]:
# Test: Verify the DataWriter changes are in place
import inspect
from highstreets.data_source_sink.datawriter import DataWriter

source = inspect.getsource(DataWriter.append_chunk)
assert "method='multi'" in source, "append_chunk is missing method='multi'"
assert "chunksize=5000" in source, "append_chunk is missing chunksize=5000"
print("OK: append_chunk uses method='multi', chunksize=5000")

source2 = inspect.getsource(DataWriter.append_data_without_check)
assert "method='multi'" in source2, "append_data_without_check is missing method='multi'"
print("OK: append_data_without_check uses method='multi', chunksize=5000")

# Verify safe_append_data exists
assert hasattr(DataWriter, 'safe_append_data'), "safe_append_data method missing"
print("OK: safe_append_data method exists")

c:\Users\Administrator\AppData\Local\pypoetry\Cache\virtualenvs\highstreets-GG6qcrPy-py3.10\lib\site-packages\geopandas\_compat.py:106: UserWarning: The Shapely GEOS version (3.10.3-CAPI-1.16.1) is incompatible with the GEOS version PyGEOS was compiled with (3.10.4-CAPI-1.16.2). Conversions between both will be slow.
  warnings.warn(


OK: append_chunk uses method='multi', chunksize=5000
OK: append_data_without_check uses method='multi', chunksize=5000
OK: safe_append_data method exists


### P1c: Parallelised aggregation + cached lookups in `mcard_weekly.py` (Local code change)

**What:** 
- 8 aggregation SQL queries now run in parallel (4 workers) instead of sequentially
- Shared lookup tables (adjustment_factors, cpi_data, inner_outer_quad) are loaded once and passed to all 8 `mcard_adjust_weekly()` calls

**Where:** Already applied in local code

**Risk:** Medium — core pipeline change. Must verify output matches exactly.

**How to test:**
1. Note the current contents of the adjusted output CSVs on S3 (or download them)
2. Run the new `mcard_weekly.py` with the same data
3. Compare every output CSV byte-for-byte (or at least row counts + checksums)

In [ ]:
# Step 1: Download current production output for comparison
# Run this BEFORE deploying the new code

import fsspec
from highstreets import config

base_dir = config.BASE_DIR
adj_path = f"{base_dir}mastercard/weekly/processed/adjusted_weekly_data/"

files_to_compare = [
    "txn_bespoke.csv", "txn_bids.csv", "txn_boroughs.csv", "txn_caz.csv",
    "txn_highstreets.csv", "txn_inner_outer.csv", "txn_london.csv",
    "txn_msoas.csv", "txn_towncentres.csv",
    "yoy_bespoke.csv", "yoy_bids.csv", "yoy_boroughs.csv", "yoy_caz.csv",
    "yoy_highstreets.csv", "yoy_inner_outer.csv", "yoy_london.csv",
    "yoy_msoas.csv", "yoy_towncentres.csv",
]

# Save current production snapshots locally for comparison
import os
os.makedirs('production_baseline', exist_ok=True)

for f in files_to_compare:
    try:
        df = pd.read_csv(f"{adj_path}{f}")
        df.to_csv(f"production_baseline/{f}", index=False)
        print(f"Saved {f}: {len(df)} rows, {len(df.columns)} cols")
    except Exception as e:
        print(f"Could not read {f}: {e}")

In [ ]:
# Step 2: After running the new mcard_weekly.py, compare outputs
# Run this AFTER the new pipeline completes

import hashlib

mismatches = []
for f in files_to_compare:
    try:
        baseline = pd.read_csv(f"production_baseline/{f}")
        new_output = pd.read_csv(f"{adj_path}{f}")

        # Check row counts
        if len(baseline) != len(new_output):
            mismatches.append(f"{f}: row count {len(baseline)} -> {len(new_output)}")
            continue

        # Check column counts
        if list(baseline.columns) != list(new_output.columns):
            mismatches.append(f"{f}: columns differ")
            continue

        # Check values (with tolerance for floating point)
        numeric_cols = baseline.select_dtypes(include='number').columns
        for col in numeric_cols:
            if not baseline[col].equals(new_output[col]):
                diff = (baseline[col] - new_output[col]).abs().max()
                if diff > 0.01:  # tolerance for rounding
                    mismatches.append(
                        f"{f}.{col}: max diff = {diff}"
                    )

        if f not in str(mismatches):
            print(f"MATCH: {f}")
    except Exception as e:
        mismatches.append(f"{f}: error - {e}")

if mismatches:
    print("\nMISMATCHES FOUND:")
    for m in mismatches:
        print(f"  - {m}")
else:
    print("\nAll files match production baseline. Safe to deploy.")

---
## P2: CI/CD Pipeline

### Steps on GitHub (github.com)

1. **Add GitHub Secrets** (Settings → Secrets and variables → Actions → New repository secret):
   - `AWS_ACCESS_KEY_ID` — your AWS access key
   - `AWS_SECRET_ACCESS_KEY` — your AWS secret key
   - `GITHUB_ACCESS_TOKEN_GLAPY` — the token used to install glapy from the private repo

2. **Push the code** to `aws-hsds` branch. The workflows will trigger:
   - `test.yml` — runs linting (black, flake8) and pytest
   - `deploy.yml` — builds Docker image and pushes to all ECR repos

3. **Check the Actions tab** on GitHub to verify both workflows pass

### After CI/CD is set up

Your new workflow becomes:
```
Edit code on EC2 → git push to aws-hsds → GitHub Actions builds Docker + pushes to ECR
```
No more pulling to local machine for Docker builds!

In [ ]:
# Verify the workflow files are correct
import yaml

for wf in ['test.yml', 'deploy.yml']:
    path = f'../.github/workflows/{wf}'
    with open(path) as f:
        config = yaml.safe_load(f)
    print(f"\n--- {wf} ---")
    print(f"  Name: {config['name']}")
    print(f"  Triggers: {list(config['on'].keys())}")
    if 'push' in config['on']:
        print(f"  Push branches: {config['on']['push'].get('branches', 'all')}")
    print(f"  Jobs: {list(config['jobs'].keys())}")

In [ ]:
# Verify .env.example exists and has the expected variables
with open('../.env.example') as f:
    lines = [l.strip() for l in f if l.strip() and not l.startswith('#')]
    vars_defined = [l.split('=')[0] for l in lines]

required = ['PG_DATABASE', 'PG_USER', 'PG_PASSWORD', 'PG_HOST', 'PG_PORT',
            'CONSUMER_KEY', 'CONSUMER_SECRET', 'LDS_API_KEY',
            'GITHUB_ACCESS_TOKEN_GLAPY']

for var in required:
    status = 'OK' if var in vars_defined else 'MISSING'
    print(f"  {status}: {var}")

---
## P3: Code Cleanup

### Already applied locally:
- Deleted: `bt_read_raw.py`, `sublicense_processor.py`, `highstreets.py`
- Added `if __name__ == "__main__":` guards to all 15 pipeline scripts
- Moved unused deps (scikit-learn, xgboost, seaborn, etc.) to dev-dependencies

### Testing

In [ ]:
# Verify dead code was removed
import os

deleted_files = [
    '../highstreets/data/bt_read_raw.py',
    '../highstreets/core/processors/sublicense_processor.py',
    '../highstreets/highstreets.py',
]
for f in deleted_files:
    exists = os.path.exists(f)
    status = 'STILL EXISTS (should be deleted)' if exists else 'Deleted OK'
    print(f"  {status}: {f}")

In [ ]:
# Verify all pipeline scripts have main() guards
import ast
import glob

pipeline_dir = '../highstreets/aws_pipeline/'
py_files = glob.glob(f'{pipeline_dir}*.py')

for filepath in sorted(py_files):
    if '__init__' in filepath or 'lambda' in filepath:
        continue
    with open(filepath) as f:
        content = f.read()
    has_guard = 'if __name__' in content
    has_main = 'def main' in content
    name = os.path.basename(filepath)
    if has_guard and has_main:
        print(f"  OK: {name}")
    else:
        print(f"  MISSING: {name} (guard={has_guard}, main={has_main})")

In [ ]:
# Verify imports still work after cleanup
# These should all succeed without errors
try:
    from highstreets.data_source_sink.dataloader import DataLoader
    from highstreets.data_source_sink.datawriter import DataWriter
    from highstreets.core.sql_manager import SQLManager
    from highstreets.core.sublicense_manager import SublicenseManager
    from highstreets.data_transformation.mcard_weekly_processor import FileProcessor
    from highstreets.data_source_sink.lookup_manager import LookupManager
    print("All core imports successful")
except ImportError as e:
    print(f"Import error: {e}")

# Verify deleted module no longer importable
try:
    from highstreets.core.processors.sublicense_processor import SublicenseProcessor
    print("WARNING: SublicenseProcessor still importable (should be deleted)")
except ImportError:
    print("OK: SublicenseProcessor correctly removed")

### Testing `if __name__` Guards

The guards ensure that importing a pipeline module doesn't execute it. Before this change, `import highstreets.aws_pipeline.mcard_weekly` would run the entire Mastercard pipeline!

**Important for Batch containers:** The Batch job definitions use commands like:
```
poetry run python -m highstreets.aws_pipeline.hex_e2e
```
The `-m` flag triggers `__main__`, so the `if __name__ == "__main__":` guard correctly fires. No changes needed to Batch job definitions.

---
## P4: Data Quality (Great Expectations)

Two new GX validation modules. These are **not yet wired into the pipeline** — they are available for you to call when ready.

### Testing the validators

In [ ]:
# Test Mastercard output validator against current production data
from highstreets.great_expectations.mcard_output_validation import (
    validate_weekly_txn_output,
    validate_weekly_yoy_output,
    validate_adjustment_factors,
)

base_dir = config.BASE_DIR
adj_path = f"{base_dir}mastercard/weekly/processed/adjusted_weekly_data/"

# Test with one txn file
try:
    txn_df = pd.read_csv(f"{adj_path}txn_highstreets.csv")
    passed, failures = validate_weekly_txn_output(txn_df, "highstreets")
    print(f"Txn validation: {'PASSED' if passed else 'FAILED'}")
    if failures:
        for f in failures:
            print(f"  - {f}")
except Exception as e:
    print(f"Could not test: {e}")

# Test with one yoy file
try:
    yoy_df = pd.read_csv(f"{adj_path}yoy_highstreets.csv")
    passed, failures = validate_weekly_yoy_output(yoy_df, "highstreets")
    print(f"YoY validation: {'PASSED' if passed else 'FAILED'}")
    if failures:
        for f in failures:
            print(f"  - {f}")
except Exception as e:
    print(f"Could not test: {e}")

In [ ]:
# Test BT output validator against current production data
from highstreets.great_expectations.bt_output_validation import validate_bt_output

# Read a recent hex output from PG
try:
    with engine.connect() as conn:
        hex_df = pd.read_sql(
            text("SELECT * FROM gisapdata.bt_footfall_tfl_hex_3hourly LIMIT 10000"),
            conn
        )
    passed, failures = validate_bt_output(
        hex_df,
        dataset_name="hex_3hourly_sample",
        id_column="hex_id",
        date_column="count_date",
    )
    print(f"BT hex validation: {'PASSED' if passed else 'FAILED'}")
    if failures:
        for f in failures:
            print(f"  - {f}")
except Exception as e:
    print(f"Could not test: {e}")

### Wiring validators into the pipeline (when ready)

Once you've confirmed the validators work correctly on current data, add them to the pipeline scripts. For example, in `mcard_weekly.py`, add before the LDS upload loop:

```python
from highstreets.great_expectations.mcard_output_validation import validate_weekly_txn_output

for prefix, resources in mcard_weekly_layers.items():
    for resource in resources:
        file_key = f"{prefix}_{resource}"
        df = pd.read_csv(f"{adj_path}{file_key}.csv")
        passed, failures = validate_weekly_txn_output(df, resource)
        if not passed:
            raise Exception(f"Validation failed for {file_key}: {failures}")
```

**Recommendation:** Run the validators in WARN mode (log but don't stop) for the first 2-3 production runs to build confidence, then switch to STOP mode.

---
## P5: Retry Safety / Idempotency

### P5a: UNIQUE Constraints (pgAdmin)

**What:** Add UNIQUE constraints on `econ_busyness_bt_daily_agg_cust_raw` and `econ_busyness_bt_outage_data` to prevent silent duplicate rows.

**Where:** pgAdmin → run against production database

**Risk:** Low — but if there are **existing duplicates** in the table, the constraint will fail. Check first.

In [ ]:
# STEP 1: Check for existing duplicates BEFORE adding constraints

check_daily_dupes = """
SELECT poi_id, poi_type, count_date, time_indicator, COUNT(*) as cnt
FROM gisapdata.econ_busyness_bt_daily_agg_cust_raw
GROUP BY poi_id, poi_type, count_date, time_indicator
HAVING COUNT(*) > 1
LIMIT 20;
"""

check_outage_dupes = """
SELECT count_date, lad_name, COUNT(*) as cnt
FROM gisapdata.econ_busyness_bt_outage_data
GROUP BY count_date, lad_name
HAVING COUNT(*) > 1
LIMIT 20;
"""

with engine.connect() as conn:
    daily_dupes = pd.read_sql(text(check_daily_dupes), conn)
    outage_dupes = pd.read_sql(text(check_outage_dupes), conn)

print(f"Daily agg duplicates: {len(daily_dupes)} groups")
if len(daily_dupes) > 0:
    display(daily_dupes)
    print("\nYou must remove duplicates before adding the constraint.")
    print("Run in pgAdmin:")
    print("""DELETE FROM gisapdata.econ_busyness_bt_daily_agg_cust_raw a
USING gisapdata.econ_busyness_bt_daily_agg_cust_raw b
WHERE a.ctid < b.ctid
  AND a.poi_id = b.poi_id
  AND a.poi_type = b.poi_type
  AND a.count_date = b.count_date
  AND a.time_indicator = b.time_indicator;""")
else:
    print("  No duplicates found. Safe to add constraint.")

print(f"\nOutage duplicates: {len(outage_dupes)} groups")
if len(outage_dupes) > 0:
    display(outage_dupes)
    print("\nYou must remove duplicates before adding the constraint.")
else:
    print("  No duplicates found. Safe to add constraint.")

In [ ]:
# STEP 2: After cleaning any duplicates, read the constraint SQL
# Copy this to pgAdmin and run it

with open('../highstreets/sql/queries/bt/create_bt_constraints.sql') as f:
    print(f.read())

In [ ]:
# STEP 3: Verify constraints were created
verify_constraints = """
SELECT conname, conrelid::regclass AS table_name, pg_get_constraintdef(oid) AS definition
FROM pg_constraint
WHERE conname IN ('uq_daily_agg_cust_raw', 'uq_outage_data');
"""
with engine.connect() as conn:
    constraints = pd.read_sql(text(verify_constraints), conn)
display(constraints)

### P5b: `safe_append_data` and `bt_outage.py` validation

The `safe_append_data()` method is available in `DataWriter` but **not yet wired into the BT scripts**. It's there for you to switch to when ready:

```python
# Instead of:
data_writer.append_data_to_postgres(data, "table_name")

# Use:
data_writer.safe_append_data(data, "table_name", start_date, end_date)
```

The `bt_outage.py` now validates that `START_DATE` and `END_DATE` are set (raises `ValueError` if missing).

---
## P6: BT Scheduling (AWS Console)

### Step-by-step AWS Console setup

#### 1. Create SNS Topic
- **Service:** SNS → Topics → Create topic
- **Type:** Standard
- **Name:** `hsds-pipeline-alerts`
- After creation: Create subscription → Protocol: Email → your official email
- Confirm the subscription from your email inbox
- Note the Topic ARN (e.g., `arn:aws:sns:eu-west-2:590183914513:hsds-pipeline-alerts`)

#### 2. Create Lambda Function
- **Service:** Lambda → Create function
- **Name:** `hsds-bt-auto-scheduler`
- **Runtime:** Python 3.10
- **Architecture:** x86_64
- **Execution role:** Create new role with basic Lambda permissions
- After creation:
  - Copy contents of `highstreets/aws_pipeline/lambda/bt_scheduler.py` into the inline editor
  - Add `psycopg2` layer (use `aws-psycopg2` from Klayers or package as a layer)
  - **Configuration → Environment variables:**
    - `PG_HOST` = your RDS endpoint
    - `PG_PORT` = 5432
    - `PG_DATABASE` = your database
    - `PG_USER` = your user
    - `PG_PASSWORD` = your password
    - `BT_STEP_FUNCTION_ARN` = ARN of your BT E2E Step Function
    - `SNS_TOPIC_ARN` = ARN from step 1
  - **Configuration → General:** Timeout = 30 seconds
  - **Configuration → VPC:** Add to same VPC as RDS (needed for PG connectivity)
  - **Permissions:** Add `states:StartExecution` and `sns:Publish` to the execution role

#### 3. Create EventBridge Schedule
- **Service:** EventBridge → Schedules → Create schedule
- **Name:** `hsds-bt-weekly-trigger`
- **Schedule type:** Recurring schedule
- **Cron expression:** `cron(0 8 ? * THU *)`  (every Thursday at 08:00 UTC)
- **Target:** Lambda function → `hsds-bt-auto-scheduler`
- **Retry policy:** 0 retries (the Lambda handles its own logic)

#### 4. Test the Lambda
- In the Lambda console, create a test event with empty JSON `{}`
- Run it and check CloudWatch Logs for output
- Expected: either "Skipped (already processed)" or "Pipeline triggered"

In [ ]:
# Preview what the Lambda would compute for today
from datetime import datetime, timedelta

today = datetime.utcnow().date()
days_since_monday = today.weekday()
this_monday = today - timedelta(days=days_since_monday)
prev_monday = this_monday - timedelta(days=7)
prev_sunday = this_monday - timedelta(days=1)

print(f"Today: {today} ({today.strftime('%A')})")
print(f"Target week: {prev_monday} (Mon) to {prev_sunday} (Sun)")

# Check current max date in PG
try:
    with engine.connect() as conn:
        result = conn.execute(
            text("SELECT MAX(count_date) FROM gisapdata.bt_footfall_tfl_hex_3hourly")
        )
        max_date = result.scalar()
    print(f"Current max(count_date) in hex table: {max_date}")
    if max_date and max_date >= prev_sunday:
        print("-> Lambda would SKIP (already processed)")
    else:
        print("-> Lambda would TRIGGER Step Function")
except Exception as e:
    print(f"Could not query PG: {e}")

---
## P7: Monitoring (AWS Console)

### Step Function Failure Notifications

Add SNS notification when a Step Function execution fails:

1. **Service:** Step Functions → Select your BT pipeline state machine
2. **Monitoring → Events:** EventBridge automatically captures state changes
3. **Service:** EventBridge → Rules → Create rule:
   - **Name:** `hsds-stepfunction-failure-alert`
   - **Event pattern:**
   ```json
   {
     "source": ["aws.states"],
     "detail-type": ["Step Functions Execution Status Change"],
     "detail": {
       "status": ["FAILED", "TIMED_OUT", "ABORTED"]
     }
   }
   ```
   - **Target:** SNS topic → `hsds-pipeline-alerts` (same topic from P6)
4. Repeat for the Mastercard Step Function

### Data Freshness Lambda

Same process as P6 Lambda but using `data_freshness_check.py`:
- **Name:** `hsds-data-freshness-check`
- **EventBridge schedule:** `cron(0 9 ? * MON *)` (every Monday 09:00 UTC)
- Same env vars as the scheduler Lambda (PG + SNS)

In [ ]:
# Preview: what the freshness checker would report right now
from datetime import datetime

tables_to_check = [
    ("BT Hex 3-hourly", "bt_footfall_tfl_hex_3hourly", "count_date", 10),
    ("BT Combined 3-hourly", "econ_busyness_bt_3hourly_counts", "count_date", 10),
    ("BT Daily Agg", "econ_busyness_bt_daily_agg_cust_raw", "count_date", 10),
    ("Mcard Weekly Txn", "econ_busyness_mcard_txn", "week_start", 45),
]

today = datetime.utcnow().date()
print(f"Checking freshness as of {today}:\n")

for label, table, date_col, threshold in tables_to_check:
    try:
        with engine.connect() as conn:
            result = conn.execute(
                text(f"SELECT MAX({date_col}) FROM gisapdata.{table}")
            )
            max_date = result.scalar()
        if max_date:
            age = (today - max_date).days
            status = 'STALE' if age > threshold else 'OK'
            print(f"  {status:5s} | {label:25s} | last: {max_date} | age: {age}d (threshold: {threshold}d)")
        else:
            print(f"  EMPTY | {label:25s} | no data")
    except Exception as e:
        print(f"  ERROR | {label:25s} | {e}")

---
## P8: Housekeeping

### Docker Multi-Stage Build

The Dockerfile now uses a two-stage build:
- **Builder stage:** Installs build-essential, git, Poetry, and all production dependencies
- **Runtime stage:** Only copies the built virtualenv + app code, with only `libpq5` as system dep

The image will be significantly smaller. Test locally if Docker is available, or rely on the CI/CD pipeline to build it.

### Dependency Changes

The following were moved from production to dev-only (still installable locally with `poetry install --with dev`):
- `scikit-learn`, `scipy`, `xgboost`
- `seaborn`, `matplotlib`
- `jupyter`, `ipykernel`, `ipython`
- `dill`

**Important:** After pulling these changes, run `poetry install` locally to update your environment. The dev deps will still be installed because Poetry installs all groups by default in development mode.

In [ ]:
# Verify pyproject.toml changes
import toml

try:
    with open('../pyproject.toml') as f:
        config = toml.load(f)

    python_version = config['tool']['poetry']['dependencies']['python']
    print(f"Python version: {python_version}")

    prod_deps = list(config['tool']['poetry']['dependencies'].keys())
    dev_deps = list(config['tool']['poetry']['dev-dependencies'].keys())

    print(f"\nProduction deps: {len(prod_deps)}")
    print(f"Dev-only deps: {len(dev_deps)}")

    moved_to_dev = ['scikit-learn', 'scipy', 'xgboost', 'seaborn',
                    'matplotlib', 'jupyter', 'ipykernel', 'ipython', 'dill']
    for dep in moved_to_dev:
        in_prod = dep in prod_deps
        in_dev = dep in dev_deps
        if in_dev and not in_prod:
            print(f"  OK: {dep} is dev-only")
        elif in_prod:
            print(f"  WARNING: {dep} is still in production deps")
        else:
            print(f"  MISSING: {dep} not found in either group")
except Exception as e:
    print(f"Could not parse pyproject.toml: {e}")
    print("(toml package may not be installed — run: pip install toml)")

---
## P9: Master Lookup Tables

Two new unified lookup tables:
- **quad-to-all:** Joins all 8 quad lookup tables on `quad_id`
- **hex-to-all:** Joins all 4 hex lookup tables on `hex_id`

These are generated automatically when `bt_lookups.py` runs (at the end, after all individual lookups).

### Testing

In [ ]:
# Test the master lookup generation locally
# This reads from the existing individual lookup tables in PG

from highstreets.data_source_sink.lookup_manager import LookupManager

try:
    lm = LookupManager()

    print("Generating quad-to-all lookup...")
    quad_to_all = lm.generate_quad_to_all_lookup()
    if quad_to_all is not None:
        print(f"  Rows: {len(quad_to_all)}")
        print(f"  Columns: {list(quad_to_all.columns)}")
        print(f"  Sample:")
        display(quad_to_all.head(3))

    print("\nGenerating hex-to-all lookup...")
    hex_to_all = lm.generate_hex_to_all_lookup()
    if hex_to_all is not None:
        print(f"  Rows: {len(hex_to_all)}")
        print(f"  Columns: {list(hex_to_all.columns)}")
        print(f"  Sample:")
        display(hex_to_all.head(3))

except Exception as e:
    print(f"Error: {e}")
    import traceback
    traceback.print_exc()

---
## Deployment Checklist

Follow this order to deploy all changes to production:

### Phase 1: Database changes (pgAdmin) — No code deployment needed
- [ ] Run `create_mcard_indexes.sql` in pgAdmin
- [ ] Check for duplicates in daily_agg and outage tables (cells above)
- [ ] Clean duplicates if any, then run `create_bt_constraints.sql` in pgAdmin
- [ ] Verify indexes and constraints exist (cells above)

### Phase 2: GitHub setup — One-time
- [ ] Add GitHub Secrets: `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, `GITHUB_ACCESS_TOKEN_GLAPY`
- [ ] Push code to `aws-hsds` branch
- [ ] Verify `test.yml` workflow passes in GitHub Actions
- [ ] Verify `deploy.yml` workflow passes and images appear in ECR

### Phase 3: Test Mastercard pipeline — Compare output
- [ ] Save current production baseline (cell above)
- [ ] Trigger Mastercard Step Function with the new Docker image
- [ ] Compare outputs (cell above) — all files should match
- [ ] If mismatch: investigate, fix, re-deploy

### Phase 4: Test BT pipeline
- [ ] Trigger BT Step Function with a known week's dates
- [ ] Verify hex, MSOA, LSOA, daily, outage data matches expectations
- [ ] Verify master lookups (quad-to-all, hex-to-all) were generated

### Phase 5: AWS Console setup
- [ ] Create SNS topic `hsds-pipeline-alerts` + email subscription
- [ ] Create `hsds-bt-auto-scheduler` Lambda
- [ ] Create EventBridge schedule (every Thursday 08:00 UTC)
- [ ] Test Lambda manually from console
- [ ] Create `hsds-data-freshness-check` Lambda
- [ ] Create EventBridge schedule (every Monday 09:00 UTC)
- [ ] Add EventBridge rule for Step Function failure → SNS notification

### Phase 6: Monitor
- [ ] Watch next Thursday auto-trigger (check email for notification)
- [ ] Watch next Monday freshness check
- [ ] After 2-3 successful runs, consider wiring GX validators into pipeline

---
## Quick Reference: File Change Summary

| File | Change | Where to act |
|------|--------|--------------|
| `datawriter.py` | `method='multi'` on append_chunk, new `safe_append_data` | Local (auto-deployed via CI/CD) |
| `mcard_weekly.py` | Parallel queries, cached lookups, main() guard | Local |
| `mcard_weekly_processor.py` | Cached lookup params on `mcard_adjust_weekly` | Local |
| `mcard_3hourly.py` | main() guard | Local |
| `mcard_weekly_intl.py` | main() guard | Local |
| `hex_e2e.py`, `msoa_e2e.py`, `lsoa_e2e.py` | main() guards | Local |
| `daily_agg.py`, `bt_outage.py` | main() guard, env var validation | Local |
| `bt_lookups.py`, `mcard_lookups.py` | main() guard, master lookups | Local |
| `sublicense.py` | main() guard | Local |
| `lookup_manager.py` | `generate_quad_to_all_lookup`, `generate_hex_to_all_lookup` | Local |
| `create_mcard_indexes.sql` | New file — DB indexes | pgAdmin |
| `create_bt_constraints.sql` | New file — UNIQUE constraints | pgAdmin |
| `.github/workflows/test.yml` | Rewritten for Poetry | GitHub |
| `.github/workflows/deploy.yml` | New file — ECR deployment | GitHub |
| `dockerfile` | Multi-stage build, GITHUB_TOKEN arg | Local (built by CI/CD) |
| `.env.example` | New file | Local |
| `pyproject.toml` | Python >=3.10, deps reorganised | Local |
| `CHANGELOG.md` | Updated with all changes | Local |
| `mcard_output_validation.py` | New GX validators | Local |
| `bt_output_validation.py` | New GX validators | Local |
| `lambda/bt_scheduler.py` | New Lambda function | AWS Lambda Console |
| `lambda/data_freshness_check.py` | New Lambda function | AWS Lambda Console |
| `.cursor/rules/highstreets.mdc` | Project context file | Local (Cursor-only) |
| Dead files (3 deleted) | `bt_read_raw.py`, `sublicense_processor.py`, `highstreets.py` | Local |